[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/youtube.ipynb)

# YouTube Data API

Collect data from YouTube with the Data API v3 through the `google-api-python-client` SDK: search videos by keyword, look up a channel, fetch statistics for videos, and download the comments of a video. The four sections share one API client and are meant to be run in order.

**Setup.** Install `google-api-python-client` and `python-dotenv`. Create an
API key in the [Google Cloud console](https://console.cloud.google.com/apis/credentials)
with the YouTube Data API v3 enabled, and put it in a `.env` file next to the
notebook:

```
YOUTUBE_API_KEY=your-key
```

Never commit the `.env` file. In Google Colab there is no `.env` file, so set
the value with `os.environ["YOUTUBE_API_KEY"] = "..."` in a cell you delete
before sharing, or use Colab's Secrets panel.

Every request costs quota. The default is 100 `search().list` calls per day,
plus 10,000 units per day for every other read at 1 unit each. Every page
of results counts again. The quota resets at midnight Pacific time.
Reference: [YouTube Data API v3](https://developers.google.com/youtube/v3/docs).

## Build the client

In [ ]:
import os 

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from dotenv import load_dotenv

load_dotenv()

In [ ]:
youtube_api_key = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=youtube_api_key)

## Search videos

Search videos by keyword with `search().list`, filter by publication date, and read the first page of results. Searches have their own budget of 100 calls per day, so use them sparingly.

In [ ]:
request = youtube.search().list(
        part="snippet",
        maxResults=25,
        q="cat",
        publishedAfter="2025-09-05T00:00:00Z"
    )
response = request.execute()

In [ ]:
type(response)

In [ ]:
response.keys()

In [ ]:
response['nextPageToken']

In [ ]:
response['pageInfo']

In [ ]:
response['items'][0]

## Channel information

Look up a channel by handle with `channels().list` and read its statistics and metadata.

In [ ]:
request = youtube.channels().list(
        part="contentDetails,id,localizations,snippet,statistics,status,topicDetails",
        forHandle="MissingSemester"
    )
response = request.execute()

In [ ]:
response.keys()

In [ ]:
response['items'][0]

## Video information

Fetch metadata and statistics for one or more videos by ID with `videos().list`. Up to 50 IDs fit in one call, separated by commas.

In [ ]:
request = youtube.videos().list(
        part="snippet,statistics,contentDetails,status",
        id="Z56Jmr9Z34Q,kgII-YWo3Zw",
        maxResults=25,
    )
response = request.execute()

In [ ]:
response.keys()

In [ ]:
response['pageInfo']

In [ ]:
response['items']

## Video comments

Download the top-level comments of a video and their replies with `commentThreads().list`, one page at a time. Pass `nextPageToken` back as `pageToken` for the next page.

In [ ]:
request = youtube.commentThreads().list(
        part="id,replies,snippet",
        videoId="Z56Jmr9Z34Q",
        maxResults=25
    )
response = request.execute()

In [ ]:
response.keys()

In [ ]:
response['nextPageToken']

In [ ]:
response['pageInfo']

In [ ]:
response['items']